# RQ3, Part 3 v2: Real SHAP Explainability (Global Network dropped)

Matches the v2 feature set. Verified: Auditing Standard remains overwhelmingly dominant (SHAP importance 0.385, essentially unchanged from the earlier run's 0.374), confirming this finding is robust and independent of the Global Network issue.

**Requires `pcaob_deficiencies_raw.csv` from Part 1.**

In [1]:
!pip install -q pandas numpy scikit-learn shap || pip install -q pandas numpy scikit-learn shap --break-system-packages

In [2]:
"""
RQ3 - PART 3 v2: Real SHAP Explainability (Global Network dropped)
================================================================================
Matches the v2 feature set used in Part 2 -- Global Network removed to
recover the real, larger sample.
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder

REAL_FEATURES = ["Auditing Standard", "Inspection Type", "Country", "Inspection Year"]


def clean_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].replace("", np.nan)
    return df.dropna(how="all")


def run_shap_analysis():
    import shap

    df = pd.read_csv("pcaob_deficiencies_raw.csv")
    df = clean_empty_strings(df)
    df = df.dropna(subset=REAL_FEATURES + ["severity"])

    X_raw = df[REAL_FEATURES].astype(str)
    y = df["severity"].values

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X = encoder.fit_transform(X_raw)
    feature_names = encoder.get_feature_names_out(REAL_FEATURES)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)

    print("Computing real SHAP values on held-out test records...")
    sample_size = min(500, len(X_test))
    X_sample = X_test[:sample_size]

    explainer = shap.TreeExplainer(rf)
    shap_results = explainer.shap_values(X_sample)

    if isinstance(shap_results, list):
        shap_values = np.asarray(shap_results[1])
    elif isinstance(shap_results, np.ndarray) and shap_results.ndim == 3:
        shap_values = shap_results[:, :, 1]
    else:
        shap_values = np.asarray(shap_results)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    print("\nReal top 15 features by mean absolute SHAP value:")
    print(importance_df.head(15).to_string(index=False))

    importance_df["real_original_feature"] = importance_df["feature"].apply(
        lambda f: next((feat for feat in REAL_FEATURES if f.startswith(feat)), f)
    )

    grouped = importance_df.groupby("real_original_feature")["mean_abs_shap"].sum().sort_values(ascending=False)
    print("\nReal SHAP importance grouped by original real feature:")
    print(grouped)

    importance_df.to_csv("rq3_shap_importance_v2.csv", index=False)
    print("\nSaved rq3_shap_importance_v2.csv")


if __name__ == "__main__":
    run_shap_analysis()


Computing real SHAP values on held-out test records...

Real top 15 features by mean absolute SHAP value:
                              feature  mean_abs_shap
            Auditing Standard_AS 2201       0.077766
            Auditing Standard_AS 2301       0.070886
            Auditing Standard_AS 1301       0.047996
            Auditing Standard_AS 3101       0.042278
Inspection Type_Triennially Inspected       0.037366
   Inspection Type_Annually Inspected       0.034998
            Auditing Standard_AS 1105       0.032241
    Auditing Standard_PCAOB Rule 3211       0.021854
            Auditing Standard_AS 2315       0.020101
            Auditing Standard_AS 2501       0.017816
            Auditing Standard_AS 2810       0.015692
            Auditing Standard_AS 1215       0.008055
            Auditing Standard_AS 2110       0.007034
               Inspection Year_2019.0       0.006475
               Inspection Year_2024.0       0.004396

Real SHAP importance grouped by original real